In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

# Check nulls first
print(orders.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [2]:
# Null delivery date = order never delivered (cancelled/lost) — treat as a signal, not junk data
orders['is_delivered'] = orders['order_delivered_customer_date'].notnull()

# Convert date columns
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Core leakage metric: how late (or early) delivery was vs. estimate
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

orders = orders.drop_duplicates(subset='order_id')

print(orders[['order_id', 'is_delivered', 'delivery_delay_days']].head())

                           order_id  is_delivered  delivery_delay_days
0  e481f51cbdc54678b7cc49136f2d6af7          True                 -8.0
1  53cdb2fc8bc7dce0b6741e2150273451          True                 -6.0
2  47770eb9100c2d0c44946d9cf07ec65d          True                -18.0
3  949d5b44dbf5de918fe9c16f97b45f8a          True                -13.0
4  ad21c59c0840e6cb83a9ceb5573f8159          True                -10.0


In [3]:
master = (orders
    .merge(items, on='order_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(reviews[['order_id','review_score']], on='order_id', how='left')
    .merge(customers, on='customer_id', how='left'))

print(master.shape)
master.head()

(119143, 25)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_delay_days,...,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,-8.0,...,8.72,1.0,credit_card,1.0,18.12,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,-8.0,...,8.72,3.0,voucher,1.0,2.00,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,-8.0,...,8.72,2.0,voucher,1.0,18.59,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,-6.0,...,22.76,1.0,boleto,1.0,141.46,4.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,-18.0,...,19.22,1.0,credit_card,3.0,179.12,5.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


In [4]:
master.to_csv('../data/cleaned/master_table.csv', index=False)
print("Saved cleaned master table:", master.shape)

Saved cleaned master table: (119143, 25)


In [5]:
import sqlite3

conn = sqlite3.connect('../data/cleaned/olist.db')
master.to_sql('master_table', conn, if_exists='replace', index=False)
conn.close()
print("Data loaded into SQLite database")

Data loaded into SQLite database


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/cleaned/olist.db')

query1 = """
SELECT
  CASE WHEN delivery_delay_days > 0 THEN 'Late' ELSE 'On-time' END AS delivery_status,
  COUNT(DISTINCT order_id) AS num_orders,
  AVG(review_score) AS avg_review_score,
  SUM(payment_value) AS total_revenue
FROM master_table
GROUP BY delivery_status;
"""

result1 = pd.read_sql(query1, conn)
print(result1)

  delivery_status  num_orders  avg_review_score  total_revenue
0            Late        6535          2.253393   1.373241e+06
1         On-time       92906          4.132788   1.920642e+07


In [7]:
query2 = """
SELECT
  review_score,
  COUNT(DISTINCT customer_unique_id) AS customers,
  SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(DISTINCT customer_unique_id) AS repeat_rate
FROM (
  SELECT customer_unique_id, review_score,
         COUNT(order_id) OVER (PARTITION BY customer_unique_id) AS order_count
  FROM master_table
) t
GROUP BY review_score
ORDER BY review_score;
"""

result2 = pd.read_sql(query2, conn)
print(result2)

   review_score  customers  repeat_rate
0           NaN        757     0.541612
1           1.0      11193     0.629233
2           2.0       3104     0.579575
3           3.0       8044     0.415589
4           4.0      18788     0.332712
5           5.0      55403     0.337455


In [10]:
query2_fixed = """
WITH distinct_orders AS (
    SELECT DISTINCT order_id, customer_unique_id, review_score
    FROM master_table
),
order_counts AS (
    SELECT
        customer_unique_id,
        review_score,
        COUNT(order_id) OVER (PARTITION BY customer_unique_id) AS order_count
    FROM distinct_orders
)
SELECT
  review_score,
  COUNT(DISTINCT customer_unique_id) AS customers,
  SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(DISTINCT customer_unique_id) AS repeat_rate
FROM order_counts
GROUP BY review_score
ORDER BY review_score;
"""

result2_fixed = pd.read_sql(query2_fixed, conn)
print(result2_fixed)

   review_score  customers  repeat_rate
0           NaN        757     0.080581
1           1.0      11193     0.067096
2           2.0       3104     0.067010
3           3.0       8044     0.067504
4           4.0      18788     0.060411
5           5.0      55403     0.069870


In [11]:
query2_fixed = """
WITH distinct_orders AS (
    SELECT DISTINCT order_id, customer_unique_id, review_score
    FROM master_table
),
order_counts AS (
    SELECT
        customer_unique_id,
        review_score,
        COUNT(order_id) OVER (PARTITION BY customer_unique_id) AS order_count
    FROM distinct_orders
)
SELECT
  review_score,
  COUNT(DISTINCT customer_unique_id) AS customers,
  SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(DISTINCT customer_unique_id) AS repeat_rate
FROM order_counts
GROUP BY review_score
ORDER BY review_score;
"""

result2_fixed = pd.read_sql(query2_fixed, conn)
print(result2_fixed)

   review_score  customers  repeat_rate
0           NaN        757     0.080581
1           1.0      11193     0.067096
2           2.0       3104     0.067010
3           3.0       8044     0.067504
4           4.0      18788     0.060411
5           5.0      55403     0.069870


In [14]:
print(master.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_delay_days', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_score', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [15]:
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [16]:
# Add the products merge to bring in category names
master = master.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')

print(master.shape)
print(master.columns.tolist())

(119143, 26)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_delay_days', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_score', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name']


In [17]:
master.to_csv('../data/cleaned/master_table.csv', index=False)

conn = sqlite3.connect('../data/cleaned/olist.db')
master.to_sql('master_table', conn, if_exists='replace', index=False)
conn.close()
print("Updated master table saved (CSV + SQLite)")

Updated master table saved (CSV + SQLite)


In [18]:
conn = sqlite3.connect('../data/cleaned/olist.db')

query3 = """
SELECT
  product_category_name,
  AVG(delivery_delay_days) AS avg_delay,
  AVG(review_score) AS avg_review,
  SUM(payment_value) AS revenue_at_risk,
  COUNT(DISTINCT order_id) AS num_orders
FROM master_table
WHERE delivery_delay_days > 0
GROUP BY product_category_name
HAVING COUNT(DISTINCT order_id) > 20
ORDER BY revenue_at_risk DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, conn)
print(result3)

    product_category_name  avg_delay  avg_review  revenue_at_risk  num_orders
0            beleza_saude   9.749009    2.289222        122865.77         650
1         cama_mesa_banho  10.931677    2.196406        119461.21         689
2        moveis_decoracao  10.610544    2.340314        108624.72         449
3  informatica_acessorios  10.474206    2.313008        107787.48         417
4      relogios_presentes   9.833713    2.158879        103800.29         406
5           esporte_lazer  10.565693    2.195167         90163.44         495
6              automotivo  12.281879    2.334471         78629.58         278
7   utilidades_domesticas  11.654596    2.214286         56831.64         308
8      ferramentas_jardim  10.915541    2.293103         51289.22         225
9       moveis_escritorio  12.347826    2.294118         47675.82         101


In [19]:
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
print(category_translation.head())

    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


In [20]:
master = master.merge(category_translation, on='product_category_name', how='left')

print(master.shape)
print(master[['product_category_name', 'product_category_name_english']].head())

(119143, 27)
   product_category_name product_category_name_english
0  utilidades_domesticas                    housewares
1  utilidades_domesticas                    housewares
2  utilidades_domesticas                    housewares
3             perfumaria                     perfumery
4             automotivo                          auto


In [21]:
master.to_csv('../data/cleaned/master_table.csv', index=False)

conn = sqlite3.connect('../data/cleaned/olist.db')
master.to_sql('master_table', conn, if_exists='replace', index=False)
conn.close()
print("Updated master table saved with English category names")

Updated master table saved with English category names


In [22]:
conn = sqlite3.connect('../data/cleaned/olist.db')

query3_final = """
SELECT
  product_category_name_english,
  AVG(delivery_delay_days) AS avg_delay,
  AVG(review_score) AS avg_review,
  SUM(payment_value) AS revenue_at_risk,
  COUNT(DISTINCT order_id) AS num_orders
FROM master_table
WHERE delivery_delay_days > 0
GROUP BY product_category_name_english
HAVING COUNT(DISTINCT order_id) > 20
ORDER BY revenue_at_risk DESC
LIMIT 10;
"""

result3_final = pd.read_sql(query3_final, conn)
print(result3_final)

  product_category_name_english  avg_delay  avg_review  revenue_at_risk  \
0                 health_beauty   9.749009    2.289222        122865.77   
1                bed_bath_table  10.931677    2.196406        119461.21   
2               furniture_decor  10.610544    2.340314        108624.72   
3         computers_accessories  10.474206    2.313008        107787.48   
4                 watches_gifts   9.833713    2.158879        103800.29   
5                sports_leisure  10.565693    2.195167         90163.44   
6                          auto  12.281879    2.334471         78629.58   
7                    housewares  11.654596    2.214286         56831.64   
8                  garden_tools  10.915541    2.293103         51289.22   
9              office_furniture  12.347826    2.294118         47675.82   

   num_orders  
0         650  
1         689  
2         449  
3         417  
4         406  
5         495  
6         278  
7         308  
8         225  
9         101 

In [23]:
query4 = """
SELECT
  strftime('%Y-%m', order_purchase_timestamp) AS month,
  SUM(payment_value) AS total_revenue,
  SUM(CASE WHEN delivery_delay_days > 0 THEN payment_value ELSE 0 END) AS revenue_from_late_orders
FROM master_table
GROUP BY month
ORDER BY month;
"""

result4 = pd.read_sql(query4, conn)
print(result4)

      month  total_revenue  revenue_from_late_orders
0   2016-09         388.47                      0.00
1   2016-10       76559.05                    198.86
2   2016-12          19.62                      0.00
3   2017-01      190806.27                   3863.20
4   2017-02      351848.13                   8657.44
5   2017-03      547769.84                  24925.68
6   2017-04      512126.52                  34793.48
7   2017-05      737425.31                  22743.65
8   2017-06      613777.41                  29691.80
9   2017-07      749242.84                  30706.05
10  2017-08      884168.36                  21936.06
11  2017-09     1030141.97                  37783.09
12  2017-10     1046083.41                  37952.52
13  2017-11     1610581.21                 174684.48
14  2017-12     1060949.63                 103299.37
15  2018-01     1425461.40                  99626.55
16  2018-02     1327338.97                 188243.75
17  2018-03     1486669.64                 268